# Camada Bronze

**Bases deste notebook**
- `pesquisa_satisfacao.csv`
- `custos.csv`
- `canais.csv`

## 1. Configuração

In [ ]:
from pyspark.sql import functions as F

catalog = "vcredit"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

BASES = {
    "pesquisa_satisfacao.csv": "pesquisa_satisfacao",
    "custos.csv": "custos",
    "canais.csv": "canais",
}

print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

## 2. Criação do catálogo, schemas e volume

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catálogo, schemas e Volume configurados")

## 3. Função de ingestão Bronze

In [ ]:
def ingerir_bronze(nome_arquivo: str, nome_tabela: str) -> dict:
    caminho = f"{landing_path.rstrip('/')}/{nome_arquivo}"
    destino = f"{bronze_schema}.{nome_tabela}"

    print(f"Iniciando ingestão: {caminho}")

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .option("encoding", "UTF-8")
        .load(caminho)
    )
    df_bronze = (
        df
        .withColumn("_arquivo_origem", F.input_file_name())
        .withColumn("_data_ingestao", F.current_timestamp())
    )
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )

    quantidade = spark.table(destino).count()

    resultado = {
        "arquivo": nome_arquivo,
        "tabela": destino,
        "quantidade_registros": quantidade,
        "status": "SUCESSO",
    }

    print(f"Ingestão concluída: {destino}")
    print(f"Registros carregados: {quantidade}")
    return resultado

## 4. Executar a ingestão das três bases

In [ ]:
resultados = []

for arquivo, tabela in BASES.items():
    resultados.append(ingerir_bronze(arquivo, tabela))

display(spark.createDataFrame(resultados))

## 5. Validar tabelas e schemas

In [ ]:
for tabela in BASES.values():
    nome_completo = f"{bronze_schema}.{tabela}"

    print(f"\n===== {nome_completo} =====")
    df = spark.table(nome_completo)
    print(f"Quantidade de registros: {df.count()}")
    df.printSchema()
    display(df.limit(5))

## 6. Conferir contagens esperadas

In [ ]:
CONTAGENS_ESPERADAS = {
    "pesquisa_satisfacao": 81021,
    "custos": 131069,
    "canais": 4,
}

validacoes = []

for tabela, esperada in CONTAGENS_ESPERADAS.items():
    nome_completo = f"{bronze_schema}.{tabela}"
    obtida = spark.table(nome_completo).count()

    validacoes.append({
        "tabela": nome_completo,
        "quantidade_esperada": esperada,
        "quantidade_obtida": obtida,
        "resultado": "OK" if obtida == esperada else "VERIFICAR",
    })

display(spark.createDataFrame(validacoes))

# Consultas
display(spark.sql(f"SELECT * FROM {TARGET_SCHEMA}.pesquisa_satisfacao LIMIT 10"))
display(spark.sql(f"SELECT * FROM {TARGET_SCHEMA}.custos LIMIT 10"))
display(spark.sql(f"SELECT * FROM {TARGET_SCHEMA}.canais"))